# Constant e approximation: Absolute and Relative Error Analysis

**Exercise 1.13**

f an amount $a$ is invested at interest rate $r$, compounded $n$ times per year, then the final value $f$ at the end of one year is given by

$$
f=a\left(1+\frac{r}{n}\right)^n
$$

This is the familiar formula for compound interest.

With simple interest,

$$
n=1
$$

Typically, compounding is done quarterly,

$$
n=4
$$

or perhaps even daily,

$$
n=365
$$

Obviously, the more frequent the compounding, the greater the final amount, because more interest is paid on previous interest.

But how much difference does this frequency actually make?

Write a program that implements the compound interest formula.

Test your program using an initial investment of

$$
a=100
$$

an interest rate of

$$
r=0.05
$$

and the following values for $n$:

$$
n=1,\ 4,\ 12,\ 365
$$

Also experiment with what happens when $n$ becomes very large.

Can you find a value such that the final amount does not grow with the frequency of compounding, as it should?

This will be easy if you use single precision, but much harder if you use double precision.

Implement the compound interest formula in two different ways:

**(a)** If the programming language you use does not have an operator for exponentiation (e.g., C), then you might implement the compound interest formula using a loop that repeatedly multiplies $a$ by

$$
\left(1+\frac{r}{n}\right)
$$

for a total of $n$ times.

Even if your programming language does have an operator for exponentiation (e.g., Fortran), try implementing the compound interest formula using such a loop and print your results for the input values.

**(b)** With the functions $\exp(x)$ and $\log(x)$, the compound interest formula can also be written as

$$
f=a\exp\left(n\log\left(1+\frac{r}{n}\right)\right)
$$

Implement this formula using the corresponding built-in functions and compare your results with those for the first implementation using the loop, for the same input values.


## 1. Setup


In [2]:
import math
import pandas as pd
import numpy as np

## 2. Definición de variables, ¿Cómo obtendremos cada una?

- **Valor de referencia (potencia directa):** $f = a\left(1+\dfrac{r}{n}\right)^n$, calculado con el operador de potencia `**` de Python.
- **Implementación (a) — bucle:** el mismo resultado, pero obtenido multiplicando `a` por $\left(1+\dfrac{r}{n}\right)$ de forma repetida, $n$ veces, sin usar el operador `**`.
- **Implementación (b) — exp/log:** usando la identidad $x^n = e^{n\ln x}$, la fórmula se reescribe como $f = a\cdot\exp\!\left(n\log\left(1+\dfrac{r}{n}\right)\right)$, calculada con `math.exp()` y `math.log()`.
- **Límite teórico ($n\to\infty$):** $f_\infty = a\cdot e^{r}$, calculado con `math.exp(r)`.

Y las ya conocidas, comparando cada implementación contra el límite teórico:
- **Absolute error:** $E_a = |f_\infty - f|$.
- **Relative error:** $r = E_a / f_\infty$

In [3]:
## Definición de funciones: cómputo aproximado de f y cálculo de errores
def compound_power(a, r, n):
    return a * (1 + r / n) ** n

def compound_loop(a, r, n):
    factor = 1 + r / n
    total = a
    for _ in range(n):
        total *= factor
    return total

def compound_explog(a, r, n):
    return a * math.exp(n * math.log(1 + r / n))

def get_error_values(a, r, n):
    exact = a * math.exp(r)            # límite "real" cuando n -> infinito
    approx = compound_explog(a, r, n)  
    abs_err = abs(exact - approx)
    rel_err = abs_err / exact
    return exact, approx, abs_err, rel_err

## 3. Cálculo y visualización de resultados

- Recorreremos cada $n=1,4,12,365$, y luego valores crecientes $n=10^k;\ k=1,2,\dots,16$
- Se aplicarán las tres implementaciones (potencia, bucle, exp/log) definidas anteriormente
- Se probará además con `float32` y `float64` para observar en qué valor de $n$ el resultado deja de crecer correctamente
- Se recopilarán los datos y serán formateados como un dataframe de pandas

In [4]:
a = 100
r = 0.05
rows = []

for k in range(1, 17):
    n = 10 ** k
    exact, approx, abs_err, rel_err = get_error_values(a, r, n)
    rows.append({
        "n": n,
        "k": k,
        "f_limite": exact,
        "f_aprox": round(approx, 10),
        "Abs. Error": round(abs_err, 10),
        "Rel. Error": rel_err,
    })

df = pd.DataFrame(rows)
df

,n,k,f_limite,f_aprox,Abs. Error,Rel. Error
0,10,1,105.12711,105.114013,1.309643e-02,1.245771e-04
1,100,2,105.12711,105.125796,1.313643e-03,1.249576e-05
2,1000,3,105.12711,105.126978,1.314044e-04,1.249957e-06
3,10000,4,105.12711,105.127096,1.314080e-05,1.249992e-07
4,100000,5,105.12711,105.127108,1.313400e-06,1.249301e-08
5,1000000,6,105.12711,105.127109,1.400000e-07,1.331829e-09
6,10000000,7,105.12711,105.127110,4.510000e-08,4.288737e-10
7,100000000,8,105.12711,105.127110,4.336000e-07,4.124519e-09
8,1000000000,9,105.12711,105.127110,4.348000e-07,4.135768e-09
9,10000000000,10,105.12711,105.127110,4.349000e-07,4.136893e-09


## 4. Observaciones:

- **Error absoluto:** El error tiende a disminuir a medida que crece $n$, acercándose cada vez más al límite teórico $a\cdot e^r$. Sin embargo, ocurre algo curioso: a partir de cierto valor de $n$ (dependiendo de la precisión usada), el error absoluto **deja de disminuir e incluso vuelve a aumentar**, llegando a ser mayor que con valores de $n$ mucho más pequeños. Esto es contraintuitivo, pues un $n$ mayor debería acercarnos más al límite cuando $n\to\infty$.
- **Error relativo:** Ocurre lo mismo que con el error absoluto: el mejor acierto (menor error) se obtiene hasta cierto punto, y en adelante el error relativo empieza a crecer de nuevo.

¿Qué nos dice esto? ¿Es esta implementación del interés compuesto sensible a valores grandes de $n$? Tiene sentido si prestamos atención a lo que pasa con `factor = 1 + r/n`: para $n$ muy grande, $r/n \approx 0$, y en la aritmética de punto flotante llega un momento en que $r/n$ es tan pequeño que, al sumarlo a $1$, la máquina simplemente **redondea el resultado de vuelta a $1.0$** (porque no hay suficientes dígitos de precisión para representar la diferencia). En ese punto, `factor` deja de reflejar el interés real: la función se convierte en $a\cdot 1^n = a$, perdiendo por completo la ganancia por interés — no porque matemáticamente $n\to\infty$ elimine el interés, sino porque el redondeo de `1+r/n` a `1.0` lo hace desaparecer artificialmente. Esto es exactamente lo que el enunciado original anticipaba: **más frecuencia de composición no garantiza un resultado más preciso una vez que la aritmética de punto flotante se queda sin dígitos para distinguir $1+r/n$ de $1$.**